In [1]:
import os
import pandas as pd
from tqdm import tqdm

# Собираем stations.csv

In [3]:
path_to_extract_from = os.path.join('../_data', 'stations')

objects = os.listdir(path_to_extract_from)

In [4]:
'''
в некоторых файлах были признаки, не повторяющиеся ни в одной другой таблице ('Unnamed: 7', 'landmark', 'dateCreated').
от них просто избавился. признак города встречается только в более поздних датасетах, вероятно это связано с расширением
территории, на которой работала компания, поэтому по умолчаню поставил город Chicago
'''

stations = pd.DataFrame(columns=['id', 'name', 'city', 'latitude', 'longitude', 'dpcapacity', 'online_date'])

for obj_name in objects:
    if obj_name.endswith('.csv'):
        obj = pd.read_csv(os.path.join(path_to_extract_from, obj_name))

    else:
        obj = pd.read_excel(os.path.join(path_to_extract_from, obj_name))

    if not 'city' in obj.columns:
            obj['city'] = 'Chicago'

    if 'online date' not in obj.columns and 'online_date' not in obj.columns:
         obj['online_date'] = pd.NA

    obj = obj.drop(['Unnamed: 7', 'landmark', 'dateCreated'], axis=1, errors='ignore')

    stations = pd.concat([stations, obj])

C:\Users\user\AppData\Local\Temp\ipykernel_13352\787698988.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stations = pd.concat([stations, obj])


In [5]:
stations['online_date'] = stations['online_date'].fillna(stations['online date'])
stations = stations.drop('online date', axis=1)

In [7]:
stations.to_csv('../data/stations.csv', index=False)

# Собираем датасет поездок

In [8]:
path_to_extract_from = os.path.join('..//_data', 'trips')

objects = os.listdir(path_to_extract_from)

In [9]:
# названия однородных признаков в разных таблицах сильно варьируются. здесь мы приводим каждую таблицу к единому стандарту

column_mapping = {
    'ride_id': ['ride_id', 'trip_id', '01 - Rental Details Rental ID'],
    'started_at': ['started_at', 'starttime', 'start_time', '01 - Rental Details Local Start Time'],
    'ended_at': ['ended_at', 'stoptime', 'end_time', '01 - Rental Details Local End Time'],
    'bike_id': ['bikeid', '01 - Rental Details Bike ID', 'bike_id'],
    'trip_duration': ['tripduration', '01 - Rental Details Duration In Seconds Uncapped'],
    'start_station_id': ['start_station_id', 'from_station_id', '03 - Rental Start Station ID'],
    'start_station_name': ['start_station_name', 'from_station_name', '03 - Rental Start Station Name'],
    'end_station_id': ['end_station_id', 'to_station_id', '02 - Rental End Station ID'],
    'end_station_name': ['end_station_name', 'to_station_name', '02 - Rental End Station Name'],
    'user_type': ['member_casual', 'usertype', 'User Type'],
    'gender': ['gender', 'Member Gender'],
    'birth_year': ['birthyear', '05 - Member Details Member Birthday Year'],
    'start_lat': ['start_lat'],
    'start_lng': ['start_lng'],
    'end_lat': ['end_lat'],
    'end_lng': ['end_lng'],
    'rideable_type': ['rideable_type']
}

rename_map = {old_name: new_name for new_name, old_names in column_mapping.items() for old_name in old_names}

In [10]:
all_data = []

for obj_name in tqdm(objects):
    file_path = os.path.join(path_to_extract_from, obj_name)
    df = pd.read_csv(file_path, low_memory=False)
    df.rename(columns=rename_map, inplace=True)

    available_columns = [col for col in column_mapping.keys() if col in df.columns]
    df = df[available_columns]

    df['started_at'] = pd.to_datetime(df['started_at'])
    df['ended_at'] = pd.to_datetime(df['ended_at'])

    all_data.append(df)

trips = pd.concat(all_data, ignore_index=True)

  0%|          | 0/79 [00:00<?, ?it/s]

100%|██████████| 79/79 [02:26<00:00,  1.85s/it]


In [11]:
trips['start_station_id'] = trips['start_station_id'].astype(str)
trips['end_station_id'] = trips['end_station_id'].astype(str)
trips['ride_id'] = trips['ride_id'].astype(str)
trips['trip_duration'] = trips['trip_duration'].astype(str)

In [ ]:
'''
вроде ничего не потерял. единственное, очень много пропусков в trips
'''

os.makedirs('..//chunks', exist_ok=True)

chunk_size = 1_000_000

for i in tqdm(range(0, len(trips), chunk_size)):
    chunk = trips.iloc[i: i + chunk_size]
    chunk.to_parquet(f'..//chunks/chunk{i//chunk_size}.parquet', index=False)

100%|██████████| 44/44 [00:44<00:00,  1.00s/it]
